# Sparsifying Rate Matrices with Erdos-Renyi Edge Removal

This notebook demonstrates how to use the `sparsify` utility to turn fully-connected CTMC rate matrices into sparse ones, while guaranteeing irreducibility (unique steady state). We'll:

1. Build a dense rate matrix and visualize it
2. Sparsify it at different densities and check connectivity
3. Compare NESS and MEPS between dense and sparse systems
4. Run a batch sweep to see how sparsity affects entropy production statistics

In [ ]:
import sys, os
sys.path.append(os.path.join(os.path.dirname(os.getcwd())))

from ctmc import ContinuousTimeMarkovChain as MC
from ctmc import arrhenius_pump_generator, sparsify, is_irreducible
from ctmc import HAS_JAX

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

print(f"JAX available: {HAS_JAX}")

## 1. Start with a Dense Rate Matrix

We'll use the Arrhenius pump generator, which models a physical system with energy levels, barriers,
and nonequilibrium driving forces. This produces a fully-connected rate matrix (every state can transition to every other).

In [ ]:
np.random.seed(7)  # reproducible demo

S = 20  # number of states
n_pumps = 100  # many pumps ensures the system stays far from equilibrium even after pruning
pump_strength = 5

R_dense = arrhenius_pump_generator(S=S, N=1, n_pumps=n_pumps, pump_strength=pump_strength)
R_dense = R_dense[0]  # single matrix, shape (S, S)

print(f'Rate matrix shape: {R_dense.shape}')
print(f'Nonzero off-diagonal entries: {np.sum(R_dense[~np.eye(S, dtype=bool)] > 0)} / {S*(S-1)}')
print(f'Irreducible: {is_irreducible(R_dense)}')

In [ ]:
# Visualize the dense rate matrix
fig, ax = plt.subplots(figsize=(7, 6))
R_plot = R_dense.copy()
np.fill_diagonal(R_plot, 0)  # zero out diagonal for cleaner visualization
R_plot[R_plot == 0] = np.nan

im = ax.imshow(np.log10(R_plot), cmap='viridis', aspect='equal')
cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label('$\\log_{10}(R_{s \\to s\'})$')
ax.set_xlabel('To state $s\'$')
ax.set_ylabel('From state $s$')
ax.set_title(f'Dense rate matrix ({S} states, fully connected)')
plt.tight_layout()

## 2. Sparsify at Different Densities

The `sparsify` function uses Erdos-Renyi edge removal: each off-diagonal transition is independently
kept with probability `p`, or equivalently you can specify a target average degree `avg_degree`.

With `ensure_connected=True` (default), any disconnected components are repaired by adding back
minimal edges — guaranteeing the CTMC remains irreducible.

In [ ]:
# Sparsify at several target average degrees
degrees = [15, 8, 4, 2]
sparse_Rs = {}

for k in degrees:
    R_sp = sparsify(R_dense, avg_degree=k, ensure_connected=True, seed=42)
    off_diag = ~np.eye(S, dtype=bool)
    actual_edges = np.sum(R_sp[off_diag] > 0)
    actual_degree = actual_edges / S
    sparse_Rs[k] = R_sp
    print(f'avg_degree={k:2d}  =>  edges: {actual_edges:3d}/{S*(S-1)}  '
          f'actual avg degree: {actual_degree:.1f}  '
          f'irreducible: {is_irreducible(R_sp)}')

In [ ]:
# Side-by-side visualization of the rate matrices at different sparsity levels
fig, axes = plt.subplots(1, len(degrees) + 1, figsize=(4 * (len(degrees) + 1), 4))

all_Rs = [('Dense', R_dense)] + [(f'k={k}', sparse_Rs[k]) for k in degrees]

for ax, (label, R) in zip(axes, all_Rs):
    R_vis = R.copy()
    np.fill_diagonal(R_vis, 0)
    # Binary view: show which transitions exist
    connectivity = (R_vis > 0).astype(float)
    connectivity[connectivity == 0] = np.nan
    
    ax.imshow(connectivity, cmap='Blues', vmin=0, vmax=1, aspect='equal')
    n_edges = int(np.nansum(connectivity))
    ax.set_title(f'{label}\n{n_edges} edges')
    ax.set_xlabel('to')
    ax.set_xticks([])
    ax.set_yticks([])

axes[0].set_ylabel('from')
fig.suptitle('Transition connectivity at different sparsity levels', fontsize=13, y=1.02)
plt.tight_layout()

## 3. Compare NESS and MEPS: Dense vs. Sparse

For a single system, let's compute the steady state (NESS) and minimum entropy production state (MEPS)
for both the dense and sparse versions, and see how the distributions and entropy production change.

In [ ]:
# Compute NESS and MEPS for the dense system
machine_dense = MC(R=R_dense)
ness_dense = machine_dense.get_ness()
meps_dense = machine_dense.get_meps()
epr_ness_dense = machine_dense.get_epr(ness_dense)
epr_meps_dense = machine_dense.get_epr(meps_dense)

def fmt_epr(val):
    """Format EPR values: fixed-point for large, scientific for tiny."""
    return f'{val:.4f}' if abs(val) > 1e-4 else f'{val:.3e}'

print(f'=== Dense (k={S-1}) ===')
print(f'  EPR(NESS) = {fmt_epr(epr_ness_dense)}')
print(f'  EPR(MEPS) = {fmt_epr(epr_meps_dense)}')
print(f'  Excess    = {(epr_ness_dense/epr_meps_dense - 1)*100:.2f}%')

# Compute for a moderately sparse version
k_demo = 8
machine_sparse = MC(R=sparse_Rs[k_demo])
ness_sparse = machine_sparse.get_ness()
meps_sparse = machine_sparse.get_meps()
epr_ness_sparse = machine_sparse.get_epr(ness_sparse)
epr_meps_sparse = machine_sparse.get_epr(meps_sparse)

print(f'\n=== Sparse (k={k_demo}) ===')
print(f'  EPR(NESS) = {fmt_epr(epr_ness_sparse)}')
print(f'  EPR(MEPS) = {fmt_epr(epr_meps_sparse)}')
print(f'  Excess    = {(epr_ness_sparse/epr_meps_sparse - 1)*100:.2f}%')

In [ ]:
# Compare the distributions side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
x = np.arange(S)
width = 0.35

ax = axes[0]
ax.bar(x - width/2, ness_dense, width, label='NESS', color='steelblue', alpha=0.85)
ax.bar(x + width/2, meps_dense, width, label='MEPS', color='coral', alpha=0.85)
ax.set_xlabel('State $s$')
ax.set_ylabel('$p(s)$')
ax.set_title(f'Dense (all-to-all)\nEPR: NESS={fmt_epr(epr_ness_dense)}, MEPS={fmt_epr(epr_meps_dense)}')
ax.legend()

ax = axes[1]
ax.bar(x - width/2, ness_sparse, width, label='NESS', color='steelblue', alpha=0.85)
ax.bar(x + width/2, meps_sparse, width, label='MEPS', color='coral', alpha=0.85)
ax.set_xlabel('State $s$')
ax.set_title(f'Sparse (avg degree {k_demo})\nEPR: NESS={fmt_epr(epr_ness_sparse)}, MEPS={fmt_epr(epr_meps_sparse)}')
ax.legend()

fig.suptitle('NESS vs MEPS distributions: Dense vs Sparse', fontsize=13, y=1.02)
plt.tight_layout()

## 4. Batch Analysis: How Does Sparsity Affect Entropy Production?

Now let's use batched computation to study the statistics. We'll generate many random nonequilibrium
systems, sparsify them at different densities, and compare the scaled excess entropy production
$\sigma_{\text{NESS}} / \sigma_{\text{MEPS}} - 1$ across sparsity levels.

This is the key quantity from the paper: it measures how close the steady state comes to minimizing
entropy production.

In [ ]:
S = 25
N = 80  # batch size
n_pumps = 50
pump_strength = 4

# Generate one batch of dense rate matrices
R_batch_dense = arrhenius_pump_generator(S=S, N=N, n_pumps=n_pumps, pump_strength=pump_strength)

# Sparsify at different levels
target_degrees = [S-1, 12, 6, 3]  # S-1 is "keep everything" (p=1)
results = {}

for k in target_degrees:
    if k >= S - 1:
        R_batch = R_batch_dense.copy()
        label = 'dense'
    else:
        R_batch = sparsify(R_batch_dense, avg_degree=k, ensure_connected=True, seed=123)
        label = f'k={k}'
    
    machine = MC(R=R_batch)
    ness = machine.get_ness()
    meps = machine.get_meps()
    
    epr_ness = machine.get_epr(ness)
    epr_meps = machine.get_epr(meps)
    epr_unif = machine.get_epr(machine.get_uniform())
    epr_rand = machine.get_epr(machine.get_random_state())
    
    # Compute excess, guarding against near-zero MEPS EPR
    with np.errstate(divide='ignore', invalid='ignore'):
        excess = epr_ness / epr_meps - 1
    excess = np.where(np.isfinite(excess), excess, np.nan)
    
    results[k] = {
        'label': label,
        'epr_ness': epr_ness,
        'epr_meps': epr_meps,
        'epr_unif': epr_unif,
        'epr_rand': epr_rand,
        'excess': excess,
    }
    
    finite_excess = excess[np.isfinite(excess)]
    if len(finite_excess) > 0:
        print(f'{label:>8s}:  median excess EPR = {np.median(finite_excess):.4f}  '
              f'mean EPR(NESS) = {epr_ness.mean():.3f}  '
              f'mean EPR(MEPS) = {epr_meps.mean():.3f}  '
              f'({len(finite_excess)}/{N} finite)')
    else:
        print(f'{label:>8s}:  all excess EPR values non-finite  '
              f'mean EPR(NESS) = {epr_ness.mean():.3f}  '
              f'mean EPR(MEPS) = {epr_meps.mean():.3f}')

In [ ]:
# Plot 1: Scatter of EPR(MEPS) vs scaled excess for each sparsity level
fig, axes = plt.subplots(1, len(target_degrees), figsize=(5 * len(target_degrees), 5),
                         sharey=True, sharex=True)

colors = {'ness': 'steelblue', 'unif': 'forestgreen', 'rand': 'darkorange'}

for ax, k in zip(axes, target_degrees):
    r = results[k]
    meps_epr = r['epr_meps']
    
    for key, color in colors.items():
        with np.errstate(divide='ignore', invalid='ignore'):
            scaled = r[f'epr_{key}'] / meps_epr - 1
        # Only plot finite positive values (log scale requires > 0)
        valid = np.isfinite(scaled) & (scaled > 0)
        ax.scatter(meps_epr[valid], scaled[valid], alpha=0.5, s=20, c=color, label=key)
    
    ax.set_yscale('log')
    ax.set_title(r['label'])
    ax.set_xlabel('$\\sigma_{\\mathrm{MEPS}}$')

axes[0].set_ylabel('$\\sigma / \\sigma_{\\mathrm{MEPS}} - 1$')
axes[-1].legend(loc='upper right')
fig.suptitle(f'Scaled excess EPR across sparsity levels (S={S}, N={N})',
             fontsize=13, y=1.02)
plt.tight_layout()

In [ ]:
# Plot 3: Heatmap of NESS distributions across batch, dense vs sparse
k_compare = 3

# Recompute for side-by-side comparison of the NESS distributions themselves
machine_d = MC(R=R_batch_dense)
ness_d = machine_d.get_ness()

R_sp = sparsify(R_batch_dense, avg_degree=k_compare, ensure_connected=True, seed=123)
machine_s = MC(R=R_sp)
ness_s = machine_s.get_ness()

# Sort both by entropy production for visual coherence
epr_d = machine_d.get_epr(ness_d)
epr_s = machine_s.get_epr(ness_s)
order_d = np.argsort(epr_d)
order_s = np.argsort(epr_s)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

im0 = axes[0].imshow(ness_d[order_d], aspect='auto', cmap='magma',
                       interpolation='nearest')
axes[0].set_title(f'NESS distributions — Dense')
axes[0].set_xlabel('State $s$')
axes[0].set_ylabel('System index (sorted by EPR)')
plt.colorbar(im0, ax=axes[0], shrink=0.8, label='$\\pi(s)$')

im1 = axes[1].imshow(ness_s[order_s], aspect='auto', cmap='magma',
                       interpolation='nearest')
axes[1].set_title(f'NESS distributions — Sparse (k={k_compare})')
axes[1].set_xlabel('State $s$')
axes[1].set_ylabel('System index (sorted by EPR)')
plt.colorbar(im1, ax=axes[1], shrink=0.8, label='$\\pi(s)$')

fig.suptitle(f'How sparsity changes the NESS landscape ({N} systems, S={S})',
             fontsize=13, y=1.02)
plt.tight_layout()

In [ ]:
# Plot 4: Direct comparison — EPR(NESS) dense vs sparse for each system
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: EPR scatter
ax = axes[0]
ax.scatter(epr_d, epr_s, alpha=0.6, s=30, c='steelblue', edgecolors='k', linewidths=0.3)
lims = [0, max(epr_d.max(), epr_s.max()) * 1.1]
ax.plot(lims, lims, 'k--', alpha=0.3, label='$y=x$')
ax.set_xlabel('$\\sigma_{\\mathrm{NESS}}$ (dense)')
ax.set_ylabel(f'$\\sigma_{{\\mathrm{{NESS}}}}$ (sparse, k={k_compare})')
ax.set_title('Steady-state EPR: dense vs sparse')
ax.legend()
ax.set_aspect('equal')

# Right: excess EPR comparison (filter to finite positive values for log-log)
ax = axes[1]
excess_d = results[S-1]['excess']
excess_s = results[k_compare]['excess']
valid = np.isfinite(excess_d) & np.isfinite(excess_s) & (excess_d > 0) & (excess_s > 0)
ax.scatter(excess_d[valid], excess_s[valid], alpha=0.6, s=30, c='coral',
           edgecolors='k', linewidths=0.3)
if valid.any():
    lims = [min(excess_d[valid].min(), excess_s[valid].min()) * 0.5,
            max(excess_d[valid].max(), excess_s[valid].max()) * 2]
    ax.plot(lims, lims, 'k--', alpha=0.3, label='$y=x$')
ax.set_xlabel('Excess EPR (dense)')
ax.set_ylabel(f'Excess EPR (sparse, k={k_compare})')
ax.set_title('$\\sigma_{\\mathrm{NESS}}/\\sigma_{\\mathrm{MEPS}} - 1$')
ax.set_xscale('log')
ax.set_yscale('log')
ax.legend()

plt.tight_layout()

## 5. Connectivity Diagnostics

At very low densities, the Erdos-Renyi mask will disconnect the graph. The `ensure_connected` option
adds back the minimum edges needed. Let's see how often repair is needed and how many edges get added.

In [ ]:
S_test = 30
R_test = arrhenius_pump_generator(S=S_test, N=1, n_pumps=40, pump_strength=5)[0]

test_degrees = np.arange(1, S_test-1, 1)
n_trials = 50

frac_disconnected = []

for k in test_degrees:
    n_disc = 0
    for trial in range(n_trials):
        R_sp = sparsify(R_test, avg_degree=k, ensure_connected=False, seed=trial*1000)
        if not is_irreducible(R_sp):
            n_disc += 1
    frac_disconnected.append(n_disc / n_trials)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(test_degrees, frac_disconnected, 'o-', color='firebrick', markersize=4)
ax.axhline(0, ls='-', c='gray', alpha=0.3)
ax.set_xlabel('Target average degree $k$')
ax.set_ylabel('Fraction disconnected\n(without repair)')
ax.set_title(f'When does sparsification break connectivity? (S={S_test}, {n_trials} trials each)')

# Mark the theoretical Erdos-Renyi threshold: k ~ ln(S)
k_threshold = np.log(S_test)
ax.axvline(k_threshold, ls='--', c='steelblue', alpha=0.7,
           label=f'$\\ln(S) = {k_threshold:.1f}$ (ER threshold)')
ax.legend()
plt.tight_layout()

## Key Takeaways

- `sparsify()` composes cleanly with any generator — just pass the dense output through it before creating the CTMC
- Parameterize via `avg_degree` (intuitive) or `p` (standard Erdos-Renyi)
- `ensure_connected=True` guarantees a unique steady state by repairing disconnected graphs with minimal added edges
- The connectivity threshold follows the classic Erdos-Renyi result: graphs become reliably connected around $k \sim \ln(S)$
- All thermodynamic quantities (EPR, NESS, MEPS) work seamlessly on sparse matrices through the existing forbidden-transition machinery